In [1]:
import sys
sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import poisson
from src.plots import plot_poisson_fit

In [2]:
matches = pd.read_csv("../data/processed/matches.csv", parse_dates = ["date"], dtype = {"season": str})
matches.shape

(3110, 16)

In [3]:
matches["match_id"] = matches.index
matches

,date,Home_team,Away_team,Home_goals,Away_goals,Result,Home_shots,Away_shots,Home_shots_on_target,Away_shots_on_target,Home_corners,Away_corners,Home_odd,Draw_odd,Away_odd,season,match_id
0,2018-08-10,Man United,Leicester,2,1,H,8,13,6,4,2,5,1.57,3.9,7.50,1819,0
1,2018-08-11,Bournemouth,Cardiff,2,0,H,12,10,4,1,7,4,1.90,3.6,4.50,1819,1
2,2018-08-11,Fulham,Crystal Palace,0,2,A,15,10,6,9,5,5,2.50,3.4,3.00,1819,2
3,2018-08-11,Huddersfield,Chelsea,0,3,A,6,13,1,4,2,5,6.50,4.0,1.61,1819,3
4,2018-08-11,Newcastle,Tottenham,1,2,A,15,15,2,5,3,5,3.90,3.5,2.04,1819,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3105,2026-10-18,Leeds,Man United,1,1,D,12,13,5,3,2,3,2.35,3.6,2.90,2627,3105
3106,2026-10-18,Nott'm Forest,Arsenal,0,1,A,14,9,2,1,8,6,6.00,3.9,1.60,2627,3106
3107,2026-10-18,Bournemouth,Sunderland,2,0,H,29,3,6,0,5,3,2.05,3.5,3.60,2627,3107
3108,2026-10-18,Brighton,Crystal Palace,5,0,H,17,5,9,1,8,5,1.53,4.5,5.50,2627,3108


is home = 1   -> from home perspective
is home = 0   -> from away perspective

In [4]:
home = pd.DataFrame({
    "match_id": matches["match_id"],
    "date" : matches["date"],
    "season" : matches["season"],
    "team": matches["Home_team"],
    "opponent": matches["Away_team"],
    "is_home": 1,  # from home team perspective
    "corners_for": matches["Home_corners"],
    "corners_against": matches["Away_corners"],
    "shots_for": matches["Home_shots"],
    "shots_against": matches["Away_shots"],
    "win_prob": 1/ matches["Home_odd"],

})

away = pd.DataFrame({
    "match_id": matches["match_id"],
    "date" : matches["date"],
    "season" : matches["season"],
    "team": matches["Away_team"],
    "opponent": matches["Home_team"],
    "is_home": 0,    #from away team perspective
    "corners_for": matches["Away_corners"],
    "corners_against": matches["Home_corners"],
    "shots_for": matches["Away_shots"],
    "shots_against": matches["Home_shots"],
    "win_prob": 1/ matches["Away_odd"],


})

team_matches = pd.concat([home, away], ignore_index = True).sort_values(["team", "date"]).reset_index(drop=True)

team_matches["match_id"].value_counts()

match_id
7       2
13      2
24      2
38      2
40      2
       ..
2209    2
2428    2
2570    2
2851    2
2969    2
Name: count, Length: 3110, dtype: int64

In [5]:
team_matches

,match_id,date,season,team,opponent,is_home,corners_for,corners_against,shots_for,shots_against,win_prob
0,7,2018-08-12,1819,Arsenal,Man City,1,2,9,9,17,0.250000
1,13,2018-08-18,1819,Arsenal,Chelsea,0,1,5,15,24,0.222222
2,24,2018-08-25,1819,Arsenal,West Ham,1,10,2,17,13,0.735294
3,38,2018-09-02,1819,Arsenal,Cardiff,0,9,3,17,14,0.636943
4,40,2018-09-15,1819,Arsenal,Newcastle,0,4,10,12,4,0.526316
...,...,...,...,...,...,...,...,...,...,...,...
6215,2994,2026-04-25,2526,Wolves,Tottenham,1,5,5,11,11,0.222222
6216,3002,2026-05-02,2526,Wolves,Sunderland,1,6,3,20,10,0.322581
6217,3010,2026-05-09,2526,Wolves,Brighton,0,1,7,5,14,0.105263
6218,3026,2026-05-17,2526,Wolves,Fulham,1,3,6,11,13,0.263158


In [6]:
team_matches[team_matches["match_id"] == 7]

,match_id,date,season,team,opponent,is_home,corners_for,corners_against,shots_for,shots_against,win_prob
0,7,2018-08-12,1819,Arsenal,Man City,1,2,9,9,17,0.250000
3594,7,2018-08-12,1819,Man City,Arsenal,0,9,2,17,9,0.512821


In [7]:
team_matches.head(15)

,match_id,date,season,team,opponent,is_home,corners_for,corners_against,shots_for,shots_against,win_prob
0,7,2018-08-12,1819,Arsenal,Man City,1,2,9,9,17,0.250000
1,13,2018-08-18,1819,Arsenal,Chelsea,0,1,5,15,24,0.222222
2,24,2018-08-25,1819,Arsenal,West Ham,1,10,2,17,13,0.735294
3,38,2018-09-02,1819,Arsenal,Cardiff,0,9,3,17,14,0.636943
4,40,2018-09-15,1819,Arsenal,Newcastle,0,4,10,12,4,0.526316
5,58,2018-09-23,1819,Arsenal,Everton,1,5,9,9,9,0.694444
6,66,2018-09-29,1819,Arsenal,Watford,1,6,6,9,13,0.694444
7,78,2018-10-07,1819,Arsenal,Fulham,0,2,4,9,21,0.602410
8,89,2018-10-22,1819,Arsenal,Leicester,1,6,4,19,8,0.653595
9,97,2018-10-28,1819,Arsenal,Crystal Palace,0,4,6,7,16,0.571429


In [8]:
WINDOW = 5
stat_cols = ["corners_for", "corners_against", "shots_for", "shots_against"]

for col in stat_cols:
    team_matches[f"{col}_avg{WINDOW}"] = (team_matches.groupby("team")[col].
                                          transform(lambda x: x.shift(1).rolling(WINDOW, min_periods= 3).mean()))  # we need to shift by

arsenal_matches = team_matches[team_matches["team"] == "Arsenal"]

In [9]:
arsenal_matches[["date", "corners_for", "corners_for_avg5"]].head(10)

,date,corners_for,corners_for_avg5
0,2018-08-12,2,NaN
1,2018-08-18,1,NaN
2,2018-08-25,10,NaN
3,2018-09-02,9,4.333333
4,2018-09-15,4,5.500000
5,2018-09-23,5,5.200000
6,2018-09-29,6,5.800000
7,2018-10-07,2,6.800000
8,2018-10-22,6,5.200000
9,2018-10-28,4,4.600000


In [10]:
team_matches.head(2)

,match_id,date,season,team,opponent,is_home,corners_for,corners_against,shots_for,shots_against,win_prob,corners_for_avg5,corners_against_avg5,shots_for_avg5,shots_against_avg5
0,7,2018-08-12,1819,Arsenal,Man City,1,2,9,9,17,0.250000,NaN,NaN,NaN,NaN
1,13,2018-08-18,1819,Arsenal,Chelsea,0,1,5,15,24,0.222222,NaN,NaN,NaN,NaN


In [11]:
avg_cols = [f"{col}_avg{WINDOW}" for col in stat_cols]

opponent_stats = team_matches[["match_id", "team"] + avg_cols].rename(columns={"team": "opponent", **{col: f"opp_{col}" for col in avg_cols}})

features = team_matches.merge(opponent_stats, on = ["match_id", "opponent"] ,how = 'left')


opponent_stats.tail()




,match_id,opponent,opp_corners_for_avg5,opp_corners_against_avg5,opp_shots_for_avg5,opp_shots_against_avg5
6215,2994,Wolves,3.0,5.4,9.4,15.6
6216,3002,Wolves,4.0,5.4,9.8,15.0
6217,3010,Wolves,4.8,3.8,13.0,13.2
6218,3026,Wolves,4.0,5.0,11.4,14.0
6219,3032,Wolves,4.2,5.2,10.8,13.0


In [12]:
len(features)

features[features["match_id"] == 65]

,match_id,date,season,team,opponent,is_home,corners_for,corners_against,shots_for,shots_against,win_prob,corners_for_avg5,corners_against_avg5,shots_for_avg5,shots_against_avg5,opp_corners_for_avg5,opp_corners_against_avg5,opp_shots_for_avg5,opp_shots_against_avg5
1599,65,2018-09-29,1819,Chelsea,Liverpool,1,4,4,10,13,0.357143,6.0,3.2,19.6,8.2,5.6,4.8,15.4,8.8
3251,65,2018-09-29,1819,Liverpool,Chelsea,0,4,4,13,10,0.393701,5.6,4.8,15.4,8.8,6.0,3.2,19.6,8.2


In [13]:
feature_cols = ["is_home", "win_prob", "corners_for_avg5", "corners_against_avg5", "shots_for_avg5", "shots_against_avg5",
                "opp_corners_for_avg5", "opp_corners_against_avg5", "opp_shots_for_avg5", "opp_shots_against_avg5"]

target_col = "corners_for"

lack of data for every team in season 18/19, and for expansions team in every season.
What's more, when team relagate and promote in next 2 year, last time the model calculated the running average was from last 5 matches in premier league before relegation

In [14]:
features[feature_cols].isna().sum()

is_home                      0
win_prob                     0
corners_for_avg5            96
corners_against_avg5        96
shots_for_avg5              96
shots_against_avg5          96
opp_corners_for_avg5        96
opp_corners_against_avg5    96
opp_shots_for_avg5          96
opp_shots_against_avg5      96
dtype: int64

In [28]:
model_data = features.dropna(subset = feature_cols)
print(f"Before: {len(features)}, After: {len(model_data)}, Deleted {len(features) - len(model_data)}, Deleted percent {(len(features) - len(model_data))/len(features)*100:.2f}%")

Before: 6220, After: 6090, Deleted 130, Deleted percent 2.09%


In [32]:
model_data[feature_cols + [target_col]].corr()[target_col].sort_values(ascending = False)

corners_for                 1.000000
win_prob                    0.408585
opp_shots_against_avg5      0.210578
opp_corners_against_avg5    0.188058
shots_for_avg5              0.180409
is_home                     0.156621
corners_for_avg5            0.154404
corners_against_avg5       -0.131400
opp_corners_for_avg5       -0.149199
shots_against_avg5         -0.159495
opp_shots_for_avg5         -0.166387
Name: corners_for, dtype: float64

correlation between target_coll (corners for) and every feature in feature_cols.
Bookmaker odds have the strongest correlation, There are many features that are calculated in odd which I have and I don't have in stats like players form, injuries, match intensity, squad rotation.